# Climate Risk Modeling for Supply Chain Resiliency
## Pipeline 01: End-to-End Data Cleaning, Normalization & Chronological Splitting

This notebook provides a complete, unified data preprocessing pipeline from raw UCI El Niño archives to clean, binned, split, and normalized datasets ready for Feature Engineering and Modeling.

Both datasets from the UCI repository are processed:
1. **`tao-all2`**: The full 18-year dataset (1980–1998) containing 178,000+ raw observations across 70 mooring stations.
2. **`elnino`**: The 14-day sample dataset containing 700+ observations across 54 buoys.

### Pipeline Stages:
1. **Raw Extraction & Ingestion**: Unzip archives and load `.gz` datasets using `.col` schema mappings.
2. **Sensor Cleaning & Imputation**: Parse dates, drop missing target values, impute transient gaps.
3. **Buoy Deployment Tagging**: Assign `buoy_segment_id` to prevent cross-buoy temporal leakage.
4. **Spherical Mooring Site Binning**: Map drifting coordinates to the 70 official NOAA PMEL TAO mooring stations using spherical Haversine distance.
5. **Target Variable Derivation**: Compute monthly climatological baselines, calculate Sea Surface Temperature Anomalies (SSTAs), and categorize risk classes.
6. **Chronological Splitting**: Split without temporal leakage (`tao-all2` by year; `elnino` by day).
7. **Leak-Free Feature Normalization**: Fit `StandardScaler` strictly on the training set and transform validation/test sets.
8. **Export**: Save final preprocessed datasets to `data/processed/`.

In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Set random seed for reproducibility
np.random.seed(42)
print("Libraries loaded successfully.")

## Step 1: Raw Extraction & Ingestion (Both Datasets)
Extract `data/el+nino.zip` into `data/raw/` and read column definitions from `.col` files.

In [ ]:
raw_dir = "data/raw"
processed_dir = "data/processed"
zip_path = "data/el+nino.zip"

os.makedirs(raw_dir, exist_ok=True)
os.makedirs(processed_dir, exist_ok=True)

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(raw_dir)
    print("[OK] Extracted zip archive to", raw_dir)

COLUMN_NAME_MAP = {
    "zon.winds": "zonal_wind",
    "mer.winds": "meridional_wind",
    "air temp.": "air_temp",
    "s.s.temp.": "ss_temp",
}

def load_uci_dataset(prefix):
    col_file = os.path.join(raw_dir, f"{prefix}.col")
    with open(col_file, 'r') as f:
        raw_cols = [line.strip() for line in f if line.strip()]
    cols = [COLUMN_NAME_MAP.get(c, c) for c in raw_cols]
    
    gz_file = os.path.join(raw_dir, f"{prefix}.dat.gz")
    if not os.path.exists(gz_file):
        gz_file = os.path.join(raw_dir, f"{prefix}.gz")
        
    df = pd.read_csv(gz_file, compression='gzip', sep=r'\s+', header=None, names=cols, na_values=['.'])
    print(f"Loaded {prefix}: {df.shape[0]} rows, {df.shape[1]} columns.")
    return df

df_tao = load_uci_dataset("tao-all2")
df_elnino = load_uci_dataset("elnino")

## Step 2: Cleaning & Missing Value Imputation
* **Date Standardization (`tao-all2`)**: Parse 2-digit years (`YYMMDD`) into standard datetime (`YYYY-MM-DD`).
* **Target Filtering**: Drop rows missing `ss_temp`.
* **Imputation**: Forward/backward fill short gaps (<= 3 days); impute remaining with monthly means.
* **Humidity Handling**: Set pre-1989 missing values to 0 (sensors not yet deployed); impute post-1989 gaps by monthly mean.

In [ ]:
# --- 1. Clean tao-all2 ---
df_tao['standardized_date'] = pd.to_datetime(df_tao['date'].astype(str), format='%y%m%d')
df_tao['year'] = df_tao['standardized_date'].dt.year
df_tao.drop(columns=['date'], inplace=True)

# Drop rows missing sea surface temperature
df_tao = df_tao.dropna(subset=['ss_temp']).reset_index(drop=True)

# Impute wind and air temperature
features = ['zonal_wind', 'meridional_wind', 'air_temp']
df_tao[features] = df_tao[features].ffill(limit=3).bfill(limit=1)
mean_by_month = df_tao.groupby('month')[features].transform('mean')
df_tao[features] = df_tao[features].fillna(mean_by_month)

# Humidity handling
monthly_humidity_mean = df_tao.groupby('month')['humidity'].transform('mean')
post_1989 = df_tao['year'] >= 1989
df_tao.loc[post_1989, 'humidity'] = df_tao.loc[post_1989, 'humidity'].fillna(monthly_humidity_mean)
df_tao['humidity'] = np.where(df_tao['year'] < 1989, 0, df_tao['humidity'])
df_tao = df_tao.round(2)

# --- 2. Clean elnino (sample dataset) ---
df_elnino = df_elnino.dropna(subset=['ss_temp']).reset_index(drop=True)
elnino_features = ['zonal_wind', 'meridional_wind', 'air_temp', 'humidity']
df_elnino[elnino_features] = df_elnino.groupby('buoy')[elnino_features].ffill(limit=3).bfill(limit=1)
df_elnino[elnino_features] = df_elnino[elnino_features].fillna(df_elnino[elnino_features].mean())
df_elnino = df_elnino.round(2)

print(f"Cleaned tao-all2: {df_tao.shape}, remaining nulls: {df_tao.isna().sum().sum()}")
print(f"Cleaned elnino:   {df_elnino.shape}, remaining nulls: {df_elnino.isna().sum().sum()}")

## Step 3: Continuous Buoy Deployment Tagging (`buoy_segment_id`)
* In `tao-all2`, readings are stacked buoy-by-buoy. We detect backward date jumps (`date_dt.diff() < 0`) to tag 73 continuous deployment segments.
* In `elnino`, buoys are already labeled with the `buoy` column (`1` to `54`).

In [ ]:
# Tag tao-all2
date_dt = pd.to_datetime(df_tao['standardized_date'])
is_new_buoy = date_dt.diff() < pd.Timedelta(days=0)
df_tao['buoy_segment_id'] = is_new_buoy.cumsum()

# Tag elnino
df_elnino['buoy_segment_id'] = df_elnino['buoy']

print(f"tao-all2 buoy segments: {df_tao['buoy_segment_id'].nunique()} (IDs 0–72)")
print(f"elnino buoy segments:   {df_elnino['buoy_segment_id'].nunique()} (IDs 1–54)")

## Step 4: Spherical Mooring Site Binning (`site_id`)
Buoy moorings sway within a ~15 km watch circle. Using spherical Great Circle (Haversine) distance with antimeridian (-180 / +180) wraparound, we snap each drifting observation to its nominal anchor station among the **70 official NOAA PMEL TAO mooring stations** (McPhaden et al., 1998).

In [ ]:
OFFICIAL_TAO_SITES = [
    # Equator (0N)
    (0, -110), (0, -125), (0, -140), (0, -155), (0, -170), (0, 180),
    (0, 170), (0, 165), (0, 160), (0, 156), (0, 147), (0, 143), (0, -95),
    # 2N
    (2, -110), (2, -125), (2, -140), (2, -155), (2, -170), (2, 180),
    (2, 165), (2, 156), (2, 147), (2, 137), (2, -95),
    # 2S
    (-2, -110), (-2, -125), (-2, -140), (-2, -155), (-2, -170), (-2, 180),
    (-2, 165), (-2, 156), (-2, -95),
    # 5N
    (5, -110), (5, -125), (5, -140), (5, -155), (5, -170), (5, 180),
    (5, 165), (5, 156), (5, 147), (5, 137), (5, -95),
    # 5S
    (-5, -110), (-5, -125), (-5, -140), (-5, -155), (-5, -170), (-5, 180),
    (-5, 165), (-5, 156), (-5, -95),
    # 8N & 9N
    (8, -110), (8, -125), (8, -140), (8, -155), (8, -170), (8, 180),
    (8, 165), (8, 156), (8, -95), (9, -140),
    # 8S
    (-8, -110), (-8, -125), (-8, -155), (-8, -170), (-8, 180),
    (-8, 165), (-8, -95)
]
sites_arr = np.array(OFFICIAL_TAO_SITES)

def haversine_vectorized(lat1, lon1, lat2_arr, lon2_arr):
    R = 6371.0  # Earth radius in km
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2_arr)
    dphi = np.radians(lat2_arr - lat1)
    dlambda = np.radians((lon2_arr - lon1 + 180.0) % 360.0 - 180.0)
    a = np.sin(dphi / 2.0)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0)**2
    return 2.0 * R * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))

def bin_dataset_coordinates(df):
    unique_coords = df[['latitude', 'longitude']].drop_duplicates().copy()
    def snap_nearest(row):
        dists = haversine_vectorized(row['latitude'], row['longitude'], sites_arr[:, 0], sites_arr[:, 1])
        best_idx = np.argmin(dists)
        nom_lat, nom_lon = sites_arr[best_idx]
        return nom_lat, nom_lon, f"{int(nom_lat)}_{int(nom_lon)}"
    
    results = unique_coords.apply(snap_nearest, axis=1)
    unique_coords['nominal_lat'] = [r[0] for r in results]
    unique_coords['nominal_lon'] = [r[1] for r in results]
    unique_coords['site_id'] = [r[2] for r in results]
    return df.merge(unique_coords, on=['latitude', 'longitude'], how='left')

df_tao = bin_dataset_coordinates(df_tao)
df_elnino = bin_dataset_coordinates(df_elnino)

print(f"Binned tao-all2 into {df_tao['site_id'].nunique()} unique mooring sites.")
print(f"Binned elnino   into {df_elnino['site_id'].nunique()} unique mooring sites.")

## Step 5: Deriving the Target Variable (SST Anomaly & Climate Risk Classes)
* **Why Climatology?** Sea surface temperature has a natural summer/winter solar cycle. El Niño is defined by departures from normal for that specific buoy location and month.
* **Formula**: `sst_anomaly = ss_temp - monthly_climatological_mean`
* **Risk Classes**: Aligned with NOAA ONI standard tiers:
  * Extreme Warm: anomaly >= +1.5 °C
  * Moderate Warm: +0.5 °C <= anomaly < +1.5 °C
  * Neutral: -0.5 °C < anomaly < +0.5 °C
  * Moderate Cool: -1.5 °C < anomaly <= -0.5 °C
  * Extreme Cool: anomaly <= -1.5 °C

In [ ]:
# For tao-all2 (contains 18 years and all 12 calendar months)
df_tao['monthly_clim'] = df_tao.groupby(['site_id', 'month'])['ss_temp'].transform('mean').round(2)
df_tao['sst_anomaly'] = (df_tao['ss_temp'] - df_tao['monthly_clim']).round(2)

def classify_risk(anomaly):
    if anomaly >= 1.5:
        return 'Extreme Warm'
    elif anomaly >= 0.5:
        return 'Moderate Warm'
    elif anomaly <= -1.5:
        return 'Extreme Cool'
    elif anomaly <= -0.5:
        return 'Moderate Cool'
    else:
        return 'Neutral'

df_tao['risk_class'] = df_tao['sst_anomaly'].apply(classify_risk)

# For elnino sample (14 days only, group by site)
df_elnino['site_clim'] = df_elnino.groupby('site_id')['ss_temp'].transform('mean').round(2)
df_elnino['sst_anomaly'] = (df_elnino['ss_temp'] - df_elnino['site_clim']).round(2)
df_elnino['risk_class'] = df_elnino['sst_anomaly'].apply(classify_risk)

print("tao-all2 Risk Class Distribution:")
print(df_tao['risk_class'].value_counts())
print("\nelnino Risk Class Distribution:")
print(df_elnino['risk_class'].value_counts())

## Step 6: Leak-Free Chronological Splitting (Both Datasets)
* **`tao-all2`**: Split by calendar years:
  * Train: 1980–1993 (captures 4 ENSO cycles: 1982–83, 1986–87, 1988–89, 1991–92)
  * Validation: 1994–1996 (moderate 1994–95 El Niño, 1995–96 La Niña)
  * Test: 1997–1998 (held-out benchmark evaluated on the historic 1997–1998 Super El Niño)
* **`elnino`**: Split chronologically by day:
  * Train: Days 1–9
  * Validation: Days 10–11
  * Test: Days 12–14

In [ ]:
# --- Split tao-all2 ---
tao_train = df_tao[df_tao['year'] <= 1993].copy()
tao_val   = df_tao[(df_tao['year'] >= 1994) & (df_tao['year'] <= 1996)].copy()
tao_test  = df_tao[df_tao['year'] >= 1997].copy()

# --- Split elnino ---
elnino_train = df_elnino[df_elnino['day'] <= 9].copy()
elnino_val   = df_elnino[(df_elnino['day'] >= 10) & (df_elnino['day'] <= 11)].copy()
elnino_test  = df_elnino[df_elnino['day'] >= 12].copy()

print(f"tao-all2 splits -> Train: {len(tao_train)}, Val: {len(tao_val)}, Test: {len(tao_test)}")
print(f"elnino   splits -> Train: {len(elnino_train)}, Val: {len(elnino_val)}, Test: {len(elnino_test)}")

## Step 7: Leak-Free Feature Normalization (Z-Score Standardization)
We fit `StandardScaler` **strictly on each dataset's training split**, and reuse those frozen means and standard deviations to transform validation and test partitions without recalculation.

In [ ]:
numeric_cols = ['zonal_wind', 'meridional_wind', 'humidity', 'air_temp', 'ss_temp']

def apply_standardization(train_df, val_df, test_df):
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_df[numeric_cols])
    val_scaled = scaler.transform(val_df[numeric_cols])
    test_scaled = scaler.transform(test_df[numeric_cols])
    
    for i, col in enumerate(numeric_cols):
        train_df[f"{col}_scaled"] = train_scaled[:, i].round(3)
        val_df[f"{col}_scaled"] = val_scaled[:, i].round(3)
        test_df[f"{col}_scaled"] = test_scaled[:, i].round(3)
        
    return train_df, val_df, test_df, scaler

tao_train, tao_val, tao_test, tao_scaler = apply_standardization(tao_train, tao_val, tao_test)
elnino_train, elnino_val, elnino_test, elnino_scaler = apply_standardization(elnino_train, elnino_val, elnino_test)

print("[OK] Standardized both datasets using training set statistics.")
print("tao-all2 training feature means:", tao_scaler.mean_.round(2))
print("elnino   training feature means:", elnino_scaler.mean_.round(2))

## Step 8: Export Processed Datasets
Export all 6 processed partitions (Train, Val, Test for both `tao-all2` and `elnino`) into `data/processed/` ready for Feature Engineering.

In [ ]:
datasets_to_save = [
    ("tao-all2-train.csv", tao_train),
    ("tao-all2-val.csv", tao_val),
    ("tao-all2-test.csv", tao_test),
    ("elnino-train.csv", elnino_train),
    ("elnino-val.csv", elnino_val),
    ("elnino-test.csv", elnino_test),
]

for filename, d in datasets_to_save:
    out_path = os.path.join(processed_dir, filename)
    d.to_csv(out_path, index=False)
    print(f"[OK] Saved {out_path} ({len(d)} rows, {d.shape[1]} columns)")

print("\nAll datasets successfully cleaned, binned, split, and standardized!")